# M3L2 E04 - StrOutputParser: extraer el texto automaticamente

## Un solo concepto

En E03 vimos que `llm.invoke()` devuelve un `AIMessage`, no un string.

Para usar la respuesta del modelo como texto simple hay que acceder a `.content`.

**El problema**: si tienes que escribir `.content` en todos lados, es codigo repetitivo
y si LangChain cambia el formato interno, hay que buscar en todo el codigo.

**La solucion**: `StrOutputParser` hace eso automaticamente como ultimo paso de la chain.

## Necesita API key de OpenAI


In [ ]:
import os, getpass
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = StrOutputParser()


## OutputParsers: normalizar la salida del modelo (Lecture M3L2 - Seccion 8)

La lecture M3L2 lista `OutputParsers` como un componente clave del pipeline:

```text
Input -> PromptTemplate -> LLM -> OutputParser -> Respuesta
                                  ^^^^^^^^^^^^
                                  Normaliza la salida del modelo
```

LangChain tiene varios parsers:

| Parser | Que hace | Cuando usarlo |
|---|---|---|
| `StrOutputParser` | Extrae el texto como string | Para respuestas de texto libre |
| `JsonOutputParser` | Parsea JSON de la respuesta | Cuando el modelo devuelve JSON |
| `PydanticOutputParser` | Valida contra un schema | Para respuestas estructuradas |
| `CommaSeparatedListOutputParser` | Lista separada por comas | Para listas simples |

En este notebook practicamos `StrOutputParser`, el mas comun.

**Conexion con M3L1**: en M3L1 la respuesta del agente era un string
que construiamos a mano con `return f"La temperatura en {ciudad} es {temp}C..."`.
No habia un paso de parsing separado.

Con LangChain, el parser es un componente independiente que:
- se puede testear solo,
- se puede reemplazar sin tocar el modelo,
- garantiza que la salida siempre tiene el mismo tipo.


## Sin parser: AIMessage con .content manual


In [ ]:
# Sin parser: resultado es AIMessage
result_raw = llm.invoke("Di solo: hola")
print(f"Tipo sin parser: {type(result_raw).__name__}")
print(f"Para obtener el texto: result_raw.content = '{result_raw.content}'")
print()
print("Si quiero pasar esto a otra funcion que espera un string, tengo que hacer .content")
print("Si la estructura interna de AIMessage cambia, mi codigo se rompe")


## Con StrOutputParser: siempre un string

`StrOutputParser` es un componente de LangChain que:
1. Recibe un `AIMessage`
2. Extrae `.content`
3. Devuelve el string

Puede usarse solo o como parte de una chain con `|`.


In [ ]:
# El parser recibe el AIMessage y devuelve el string
result_parsed = parser.invoke(result_raw)
print(f"Tipo con parser: {type(result_parsed).__name__}")
print(f"Valor: '{result_parsed}'")
print()
print("StrOutputParser convierte el AIMessage en un string simple")


## TODO 1: componer `llm | parser` con LCEL

Lo mas comun es usar el parser directamente en la chain.
Asi el resultado final siempre es un string, sin tener que llamar `.content`.


In [ ]:
# TODO 1: componer llm | parser y guardar en 'chain_con_parser'
# chain_con_parser = llm | parser
chain_con_parser = None  # reemplazar

print(f"Chain: {type(chain_con_parser).__name__ if chain_con_parser else 'TODO no completado'}")


## TODO 2: invocar y comparar tipos


In [ ]:
# TODO 2: invocar 'chain_con_parser' con "Di solo: hola"
# y verificar que el resultado es un string, no un AIMessage

# result_chain = chain_con_parser.invoke("Di solo: hola")
# print(f"Con llm solo:         tipo={type(result_raw).__name__}")
# print(f"Con llm | parser:     tipo={type(result_chain).__name__}")
# print()
# print(f"Valor: '{result_chain}'")
# print()
# print("Con el parser en la chain, el resultado final siempre es un string")


In [ ]:
def run_checks():
    assert chain_con_parser is not None, "TODO 1: chain_con_parser es None"
    result = chain_con_parser.invoke("Di solo: test")
    assert isinstance(result, str), f"Debe ser str, es {type(result).__name__}"
    assert len(result) > 0
    # Sin parser: es AIMessage
    from langchain_core.messages import AIMessage
    raw = llm.invoke("Di solo: test")
    assert isinstance(raw, AIMessage), "llm.invoke sin parser debe devolver AIMessage"
    print("M3L2 E04 checks passed")

run_checks()


## Cierre

| Sin parser | Con `StrOutputParser` |
|---|---|
| `llm.invoke(q)` → `AIMessage` | `(llm \| parser).invoke(q)` → `str` |
| Hay que escribir `.content` en todos lados | El parser lo hace automaticamente |
| Si cambia la estructura interna, se rompe | Aislado del formato interno |

**Siguiente**: E01 muestra como agregar el `PromptTemplate` para tener la chain completa.
